# Temporal Delinquency Targets (2000-2020)

This notebook replaces the standalone Python target-builder. Run all cells to create the temporally defined delinquency/contact targets from `nlsy79_child_youngadult_selected_crime_features.csv`.

The target window is **2000-2020**. For a clean baseline prediction design, predictors should come from **before 2000**, for example up to 1998.

## Target Design

The notebook creates two target families.

**Broad delinquency/contact** includes direct behavior and contact indicators: hurting someone badly, physical fighting, damaging school property, being sentenced to corrections/jail/reform, and probation.

**Justice contact** is narrower and includes only corrections/jail/reform and probation.

For each family, two binary targets are created:

- `any`: at least one positive year between 2000 and 2020
- `persistent`: positive items in at least two different years between 2000 and 2020

NLSY negative missing codes (`-1, -2, -3, -4, -5, -7`) are treated as missing. Respondents with no observed target year are left missing instead of being coded as 0.

In [ ]:
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
if HERE.name != "temporal_targets":
    HERE = HERE / "temporal_targets"
PROJECT_DIR = HERE.parent

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
TARGETS_CSV = HERE / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
ITEMS_CSV = HERE / "temporal_delinquency_target_items_2000_2020.csv"
BASE_RATES_CSV = HERE / "temporal_delinquency_target_base_rates_2000_2020.csv"

MISSING_CODES = {-1, -2, -3, -4, -5, -7}

## Target Item List

The item list below is embedded in the notebook so that the notebook can recreate the outputs without importing a separate Python script.

In [ ]:
items_csv = """csv_code,year,item_type,description
C2466100,2000,violence,hurt someone badly enough to need a doctor
C2466400,2000,property,damaged school property on purpose
Y1176200,2000,violence,physical fight at school or work
Y1176900,2000,justice,sentenced to corrections/jail/reform
Y1177000,2000,justice,on probation
C2765800,2002,violence,hurt someone badly enough to need a doctor
C2766100,2002,property,damaged school property on purpose
Y1415900,2002,violence,hurt someone badly enough to need a doctor
Y1416200,2002,property,damaged school property on purpose
Y1416800,2002,violence,physical fight at school or work
Y1417500,2002,justice,sentenced to corrections/jail/reform
Y1417600,2002,justice,on probation
C3045500,2004,violence,hurt someone badly enough to need a doctor
C3045800,2004,property,damaged school property on purpose
Y1667300,2004,violence,hurt someone badly enough to need a doctor
Y1667600,2004,property,damaged school property on purpose
Y1668200,2004,violence,physical fight at school or work
Y1668900,2004,justice,sentenced to corrections/jail/reform
Y1669000,2004,justice,on probation
C3366300,2006,violence,hurt someone badly enough to need a doctor
C3366600,2006,property,damaged school property on purpose
Y1940600,2006,violence,hurt someone badly enough to need a doctor
Y1940900,2006,property,damaged school property on purpose
Y1941500,2006,violence,physical fight at school or work
Y1942200,2006,justice,sentenced to corrections/jail/reform
Y1942300,2006,justice,on probation
C3870000,2008,violence,hurt someone badly enough to need a doctor
C3870300,2008,property,damaged school property on purpose
Y2256600,2008,violence,hurt someone badly enough to need a doctor
Y2256900,2008,property,damaged school property on purpose
Y2257500,2008,violence,physical fight at school or work
Y2258200,2008,justice,sentenced to corrections/jail/reform
Y2258300,2008,justice,on probation
C5118200,2010,violence,hurt someone badly enough to need a doctor
C5118500,2010,property,damaged school property on purpose
Y2608200,2010,violence,hurt someone badly enough to need a doctor
Y2608500,2010,property,damaged school property on purpose
Y2609100,2010,violence,physical fight at school or work
Y2609800,2010,justice,sentenced to corrections/jail/reform
Y2609900,2010,justice,on probation
C5695700,2012,violence,hurt someone badly enough to need a doctor
C5696000,2012,property,damaged school property on purpose
Y2958300,2012,violence,hurt someone badly enough to need a doctor
Y2958600,2012,property,damaged school property on purpose
Y2959200,2012,violence,physical fight at school or work
Y2959900,2012,justice,sentenced to corrections/jail/reform
Y2960000,2012,justice,on probation
C5967600,2014,violence,hurt someone badly enough to need a doctor
C5967900,2014,property,damaged school property on purpose
Y3325701,2014,violence,hurt someone badly enough to need a doctor
Y3325704,2014,property,damaged school property on purpose
Y3325801,2014,violence,physical fight at school or work
Y3326300,2014,justice,sentenced to corrections/jail/reform
Y3326400,2014,justice,on probation
Y3670801,2016,violence,hurt someone badly enough to need a doctor
Y3670804,2016,property,damaged school property on purpose
Y3670901,2016,violence,physical fight at school or work
Y3671500,2016,justice,sentenced to corrections/jail/reform
Y3671600,2016,justice,on probation
Y4275601,2018,violence,hurt someone badly enough to need a doctor
Y4275604,2018,property,damaged school property on purpose
Y4275701,2018,violence,physical fight at school or work
Y4276300,2018,justice,sentenced to corrections/jail/reform
Y4276400,2018,justice,on probation
Y4596801,2020,violence,hurt someone badly enough to need a doctor
Y4596804,2020,property,damaged school property on purpose
Y4596901,2020,violence,physical fight at school or work
Y4597500,2020,justice,sentenced to corrections/jail/reform
Y4597600,2020,justice,on probation
"""

item_table = pd.read_csv(StringIO(items_csv))
item_table["available_in_feature_file"] = False
item_table.head(12)

In [ ]:
pd.crosstab(item_table["year"], item_table["item_type"])

## Build Targets

A year is observed if at least one item in the target family is non-missing. A year is positive if at least one observed item in that year is greater than zero.

In [ ]:
def clean_value(value):
    if pd.isna(value):
        return None
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    if int(value) in MISSING_CODES:
        return None
    return value


def summarize_years(row, items, kind=None):
    if kind is not None:
        items = items[items["item_type"] == kind]

    observed_years = []
    positive_years = []
    positive_item_count = 0

    for year, year_items in items.groupby("year"):
        values = [clean_value(row.get(code)) for code in year_items["csv_code"]]
        observed = [v for v in values if v is not None]
        if observed:
            observed_years.append(int(year))
        positives = [v for v in observed if v > 0]
        if positives:
            positive_years.append(int(year))
            positive_item_count += len(positives)

    observed_years = sorted(set(observed_years))
    positive_years = sorted(set(positive_years))
    return {
        "ever": int(bool(positive_years)) if observed_years else np.nan,
        "persistent": int(len(positive_years) >= 2) if observed_years else np.nan,
        "positive_item_count": positive_item_count if observed_years else np.nan,
        "positive_year_count": len(positive_years) if observed_years else np.nan,
        "first_event_year": positive_years[0] if positive_years else np.nan,
        "second_event_year": positive_years[1] if len(positive_years) >= 2 else np.nan,
        "last_observed_year": observed_years[-1] if observed_years else np.nan,
    }


def base_rate_table(targets, target_cols):
    rows = []
    total_n = len(targets)
    for col in target_cols:
        valid = targets[col].dropna()
        rows.append({
            "target": col,
            "total_n": total_n,
            "eligible_n": len(valid),
            "missing_n": total_n - len(valid),
            "positive_n": int((valid == 1).sum()),
            "negative_n": int((valid == 0).sum()),
            "base_rate": float(valid.mean()) if len(valid) else np.nan,
        })
    return pd.DataFrame(rows)

In [ ]:
df = pd.read_csv(DATA_PATH)
item_table["available_in_feature_file"] = item_table["csv_code"].isin(df.columns)

missing_items = item_table.loc[~item_table["available_in_feature_file"]]
if len(missing_items):
    raise ValueError(f"Some target items are missing from the feature file:\n{missing_items}")

rows = []
for _, row in df.iterrows():
    broad = summarize_years(row, item_table)
    justice = summarize_years(row, item_table, kind="justice")
    rows.append({
        "C0000100": row["C0000100"],
        "C0000200": row.get("C0000200", np.nan),
        "later_any_delinquency_contact_2000_2020": broad["ever"],
        "later_persistent_delinquency_contact_2000_2020": broad["persistent"],
        "later_delinquency_contact_positive_item_count_2000_2020": broad["positive_item_count"],
        "later_delinquency_contact_positive_year_count_2000_2020": broad["positive_year_count"],
        "later_delinquency_contact_first_event_year_2000_2020": broad["first_event_year"],
        "later_delinquency_contact_second_event_year_2000_2020": broad["second_event_year"],
        "later_delinquency_contact_last_observed_year_2000_2020": broad["last_observed_year"],
        "later_any_justice_contact_2000_2020": justice["ever"],
        "later_persistent_justice_contact_2000_2020": justice["persistent"],
        "later_justice_contact_positive_item_count_2000_2020": justice["positive_item_count"],
        "later_justice_contact_positive_year_count_2000_2020": justice["positive_year_count"],
        "later_justice_contact_first_event_year_2000_2020": justice["first_event_year"],
        "later_justice_contact_second_event_year_2000_2020": justice["second_event_year"],
        "later_justice_contact_last_observed_year_2000_2020": justice["last_observed_year"],
    })

targets = pd.DataFrame(rows)
target_cols = [
    "later_any_delinquency_contact_2000_2020",
    "later_persistent_delinquency_contact_2000_2020",
    "later_any_justice_contact_2000_2020",
    "later_persistent_justice_contact_2000_2020",
]
base_rates = base_rate_table(targets, target_cols)

item_table.to_csv(ITEMS_CSV, index=False)
targets.to_csv(TARGETS_CSV, index=False)
base_rates.to_csv(BASE_RATES_CSV, index=False)

print(f"Wrote {TARGETS_CSV}")
print(f"Wrote {ITEMS_CSV}")
print(f"Wrote {BASE_RATES_CSV}")

## Check Outputs

In [ ]:
base_rates

In [ ]:
targets.head()

In [ ]:
pd.crosstab(item_table["year"], item_table["item_type"])

## Recommended Main Target

The recommended main target is:

`later_persistent_delinquency_contact_2000_2020`

It captures repeated/persistent broad delinquency or contact and has a much more usable base rate than the narrower persistent justice-contact target. For a clean baseline Book-of-Life setup, pair it with predictors observed before 2000.